# {{PROJECT_NAME}}

[![Open in JupyterLab](https://img.shields.io/badge/Open%20in-JupyterLab-F37626?logo=jupyter&logoColor=white)]({{JL_URL}})

| | |
|---|---|
| **Type** | KFP v2 eval-first fine-tuning pipeline (NeMo Customizer backend) |
| **Model** | [{{HF_MODEL_ID}}](https://huggingface.co/{{HF_MODEL_ID}}) |
| **Dataset** | [{{HF_DATASET_ID}}](https://huggingface.co/datasets/{{HF_DATASET_ID}}) |

KFP v2 eval-first fine-tuning pipeline using NeMo Microservices Customizer as training backend.

**Development workflow:**
1. Edit `config.yaml` — set model IDs, NeMo URL, datasets, thresholds, and judge prompt
2. Edit `formatters.py` — add one formatter per dataset, register in `FORMATTERS`
3. Write step logic in the `@dsl.component` cells below
4. Save (`Ctrl+S`), run the **Build → `pipeline.py`** cell
5. Trigger **Deploy to KFP** workflow (or run **Compile & Submit** below)

**Pipeline DAG:**
```
download_model
  prepare_dataset ──► baseline_eval ──► fine_tune ──► export_adapter ─┬─► post_finetune_eval ──►─┐
                  ──► baseline_safety_eval ──────►────────────────────┘                          │
                                                        └─► safety_eval ──────────────────────────►┤
                                                                                                   │
                                                                                          deployment_gate
```

> `fine_tune` runs after `baseline_safety_eval` — on single-node k3s, GPU steps cannot
> overlap without exceeding the allocatable memory limit.

## Step Development

Write step logic inside each `@dsl.component` function body.

- **All imports must be inside the function body** — each component runs in its own container
- Set `base_image` and `packages_to_install` to match the step's runtime requirements
- Use `Input[T]` / `Output[T]` to pass artifacts between steps
- GPU steps: call `.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")` in the pipeline cell

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact

### download_model

Downloads the HuggingFace base model to the cache PVC before any eval stage.
Eval stages (baseline_eval, post_finetune_eval) use the HF model directly.
The NeMo fine_tune stage uses the NeMo Customizer — it does not need the HF download.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["huggingface_hub>=0.21"],
)
def download_model(
    base_model_id: str,
):
    import os
    from huggingface_hub import snapshot_download
    print(f"Downloading {base_model_id} to HF cache...")
    snapshot_download(
        repo_id=base_model_id,
        cache_dir="/root/.cache/huggingface/hub",
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"Download complete: {base_model_id}")

### prepare_dataset

Loads datasets from HuggingFace, applies formatters from `formatters.py`, splits into train/val/test.
Output format: `{"instruction": ..., "response": ..., "source": ...}` — used by all eval stages.
The `fine_tune` stage converts to NeMo format (`{"input": ..., "output": ...}`) internally when uploading.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=[
        "datasets<3.0",
        "huggingface_hub>=0.21.2,<0.24",
        "transformers",
    ],
)
def prepare_dataset(
    dataset_names: list,
    val_size: float,
    test_size: float,
    train_out: Output[Dataset],
    val_out: Output[Dataset],
    test_out: Output[Dataset],
    shuffle_seed: int = 42,
):
    import json, pathlib, random

    # <<< FORMATTERS_INJECT >>>
    # formatters.py is inlined here by the Build cell. Do not remove this marker.

    # <<< LOADERS_INJECT >>>
    # loaders.py is inlined here by the Build cell. Do not remove this marker.

    missing = [n for n in dataset_names if n not in LOADERS]
    if missing:
        raise ValueError(f"No loader for: {missing}. Add to loaders.py LOADERS dict.")

    all_rows = []
    for name in dataset_names:
        ds = LOADERS[name]()
        all_rows.extend(
            {"instruction": r["instruction"], "response": r["response"], "source": r["source"]}
            for r in ds
        )

    random.seed(shuffle_seed)
    random.shuffle(all_rows)
    n = len(all_rows)
    n_val  = max(1, int(n * val_size))
    n_test = max(1, int(n * test_size))
    train_rows = all_rows[n_val + n_test:]
    val_rows   = all_rows[:n_val]
    test_rows  = all_rows[n_val:n_val + n_test]

    pathlib.Path(train_out.path).write_text(json.dumps(train_rows))
    pathlib.Path(val_out.path).write_text(json.dumps(val_rows))
    pathlib.Path(test_out.path).write_text(json.dumps(test_rows))
    print(f"Dataset split: {len(train_rows)} train / {len(val_rows)} val / {len(test_rows)} test")

### baseline_eval

Evaluates the base model on the validation set before fine-tuning. Uses `model.hf_id` from config.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "accelerate",
        "mlflow",
        "nvtx",
    ],
)
def baseline_eval(
    val: Input[Dataset],
    base_model_id: str,
    eval_sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 1024,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
):
    import json, pathlib, mlflow, nvtx, torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    val_data = json.loads(pathlib.Path(val.path).read_text())[:eval_sample_size]

    # <<< UTILS_INJECT >>>

    # <<< EVAL_HELPERS_INJECT >>>

    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
    model.eval()
    _infer = make_infer_fn(tokenizer, model, system_message, max_new_tokens, do_sample)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-baseline"):
        mlflow.log_param("eval_sample_size", len(val_data))
        correct = 0
        with nvtx.annotate("baseline_eval_inference"):
            for row in val_data:
                generated = _infer(row)
                # ---- USER CODE BLOCK ----
                # TODO: compare generated answer to ground truth using extract_answer()
                # if extract_answer(generated) == extract_answer(row["response"]): correct += 1
                # ---- END USER CODE BLOCK ----
            torch.cuda.synchronize()
        accuracy = correct / len(val_data) if val_data else 0.0
        mlflow.log_metric("baseline_accuracy", accuracy)

    pathlib.Path(metrics.path).write_text(json.dumps({"baseline_accuracy": accuracy}))
    print(f"Baseline accuracy: {accuracy:.4f}")

### baseline_safety_eval

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "accelerate",
        "openai",
        "mlflow",
        "nvtx",
    ],
)
def baseline_safety_eval(
    val: Input[Dataset],
    base_model_id: str,
    judge_model_id: str,
    judge_system_prompt: str,
    judge_base_url: str,
    sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 256,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
):
    import json, pathlib, mlflow, nvtx, torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from openai import OpenAI

    val_data = json.loads(pathlib.Path(val.path).read_text())[:sample_size]

    # <<< UTILS_INJECT >>>

    # ---- USER CODE BLOCK ----
    # TODO: load base model (no adapter), run inference + judge loop, collect scores
    # tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    # model = AutoModelForCausalLM.from_pretrained(
    #     base_model_id, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
    # model.eval()
    # client = OpenAI(base_url=judge_base_url, api_key="ollama")
    scores = []  # TODO: fill with float scores from judge
    avg_score = sum(scores) / len(scores) if scores else 0.0
    # ---- END USER CODE BLOCK ----

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-baseline-safety-eval"):
        mlflow.log_metric("baseline_safety_avg_score", avg_score)
        mlflow.log_metric("baseline_safety_sample_size", len(scores))
        mlflow.log_param("judge_model_id", judge_model_id)

    pathlib.Path(metrics.path).write_text(json.dumps({"baseline_safety_avg_score": avg_score}))
    print(f"Baseline safety avg score: {avg_score:.4f} ({len(scores)} samples)")

### fine_tune (NeMo Customizer)

Submits an async fine-tuning job to NeMo Microservices Customizer. Converts training data
to NeMo JSONL format, uploads to the NeMo data store, creates a job, polls for completion,
and downloads the resulting checkpoint.

See `WORKBOOK.md § 3` for the full implementation guide.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["nemo-microservices>=25.12.0", "requests", "mlflow"],
)
def fine_tune(
    train: Input[Dataset],
    base_model_id: str,
    nemo_base_url: str,
    nemo_namespace: str,
    nemo_data_store_path: str,
    num_epochs: int,
    learning_rate: float,
    batch_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    ft_model: Output[Model],
):
    import json, pathlib, time, mlflow
    from nemo_microservices import NeMoMicroservices

    train_data = json.loads(pathlib.Path(train.path).read_text())

    # Convert to NeMo training format
    nemo_rows = [{"input": r["instruction"], "output": r["response"]} for r in train_data]
    train_jsonl = "\n".join(json.dumps(r) for r in nemo_rows).encode()
    print(f"Prepared {len(nemo_rows)} training examples in NeMo JSONL format")

    client = NeMoMicroservices(base_url=nemo_base_url)

    # ---- USER CODE BLOCK ----
    # Step 1: Upload dataset to NeMo data store
    # Consult the NeMo Microservices SDK docs for the exact data store upload API.
    # Example (SDK v25.12+):
    # import io
    # upload_path = f"{nemo_data_store_path.rstrip('/')}/{run_id}-train.jsonl"
    # upload_resp = client.datastore.files.upload(path=upload_path, file=("train.jsonl", io.BytesIO(train_jsonl)))
    # dataset_ref = upload_resp.id
    dataset_ref = None  # TODO: set to uploaded dataset reference
    # ---- END USER CODE BLOCK ----

    if dataset_ref is None:
        raise RuntimeError("TODO: implement dataset upload (see WORKBOOK.md § 3)")

    # ---- USER CODE BLOCK ----
    # Step 2: Create customization job
    job = client.customization.jobs.create(
        name=f"{run_id}-finetune",
        model=base_model_id,
        training_config={
            "num_epochs": num_epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
        },
        dataset={"train": {"file_id": dataset_ref}},
    )
    print(f"Customization job submitted: {job.id}")
    # ---- END USER CODE BLOCK ----

    # Poll for completion (1h timeout)
    for _ in range(720):
        j = client.customization.jobs.retrieve(job.id)
        print(f"  {j.status}")
        if j.status in ("completed", "failed", "cancelled"):
            break
        time.sleep(5)

    if j.status != "completed":
        raise RuntimeError(f"NeMo customization job failed with status: {j.status}")

    # ---- USER CODE BLOCK ----
    # Step 3: Download checkpoint artifact to ft_model.path
    # Consult SDK docs for checkpoint download API:
    # client.customization.jobs.download(job.id, output_dir=ft_model.path)
    pathlib.Path(ft_model.path).mkdir(parents=True, exist_ok=True)
    # TODO: implement checkpoint download
    # ---- END USER CODE BLOCK ----

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-finetune"):
        mlflow.log_params({
            "nemo_job_id": job.id,
            "num_epochs": num_epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
        })

    print(f"Fine-tuning complete. Checkpoint at: {ft_model.path}")

### export_adapter

Converts the NeMo `.nemo` checkpoint to a standard HF PEFT adapter directory
(`adapter_model.safetensors` + `adapter_config.json`). This is the format expected
by `serving-vllm`'s `build-push.yaml`.

See `WORKBOOK.md § 4` for the full implementation guide.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/nemo:25.04",
    packages_to_install=[],
)
def export_adapter(
    ft_model: Input[Model],
    base_model_id: str,
    hf_adapter: Output[Model],
):
    import pathlib, subprocess, sys

    ft_path = pathlib.Path(ft_model.path)
    nemo_ckpt = next(ft_path.glob("*.nemo"), None)
    if nemo_ckpt is None:
        raise RuntimeError(f"No .nemo checkpoint found in {ft_model.path}")

    out_path = pathlib.Path(hf_adapter.path)
    out_path.mkdir(parents=True, exist_ok=True)

    # ---- USER CODE BLOCK ----
    # Convert NeMo checkpoint to HF PEFT format.
    # Locate nemo2hf.py in the container: find /opt/nemo -name "nemo2hf.py" 2>/dev/null
    # subprocess.run([
    #     "python3", "/opt/nemo/scripts/nemo2hf.py",
    #     "--input", str(nemo_ckpt),
    #     "--output", str(out_path),
    # ], check=True)
    # ---- END USER CODE BLOCK ----

    # Verify output contains expected HF PEFT files
    adapter_bin = out_path / "adapter_model.safetensors"
    adapter_cfg = out_path / "adapter_config.json"
    if not adapter_bin.exists():
        raise RuntimeError(f"export_adapter: {adapter_bin} not found — nemo2hf conversion failed")
    if not adapter_cfg.exists():
        raise RuntimeError(f"export_adapter: {adapter_cfg} not found — nemo2hf conversion failed")

    print(f"Export complete: {hf_adapter.path}")
    print(f"  adapter_model.safetensors: {adapter_bin.stat().st_size / 1e6:.1f} MB")

### post_finetune_eval

Evaluates the exported HF PEFT adapter on the validation set. Takes `hf_adapter` from
`export_adapter` (not `ft_model` directly — the adapter must be in HF PEFT format first).

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "peft>=0.13",
        "accelerate",
        "mlflow",
        "nvtx",
    ],
)
def post_finetune_eval(
    val: Input[Dataset],
    hf_adapter: Input[Model],
    base_model_id: str,
    eval_sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 1024,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
):
    import json, pathlib, mlflow, nvtx, torch

    val_data = json.loads(pathlib.Path(val.path).read_text())[:eval_sample_size]

    # <<< UTILS_INJECT >>>

    # <<< EVAL_HELPERS_INJECT >>>

    # ---- USER CODE BLOCK ----
    # Load base model + HF PEFT adapter from hf_adapter.path
    # from transformers import AutoTokenizer, AutoModelForCausalLM
    # from peft import PeftModel
    # tokenizer = AutoTokenizer.from_pretrained(hf_adapter.path)
    # model = AutoModelForCausalLM.from_pretrained(
    #     base_model_id, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
    # model = PeftModel.from_pretrained(model, hf_adapter.path)
    # model.eval()
    tokenizer = None  # TODO
    model = None      # TODO
    # ---- END USER CODE BLOCK ----

    _infer = make_infer_fn(tokenizer, model, system_message, max_new_tokens, do_sample)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-postft-eval"):
        mlflow.log_param("eval_sample_size", len(val_data))
        correct = 0
        with nvtx.annotate("post_finetune_eval_inference"):
            for row in val_data:
                generated = _infer(row)
                # ---- USER CODE BLOCK ----
                # TODO: compare generated answer to ground truth
                # if extract_answer(generated) == extract_answer(row["response"]): correct += 1
                # ---- END USER CODE BLOCK ----
            torch.cuda.synchronize()
        accuracy = correct / len(val_data) if val_data else 0.0
        mlflow.log_metric("postft_accuracy", accuracy)

    pathlib.Path(metrics.path).write_text(json.dumps({"postft_accuracy": accuracy}))
    print(f"Post-FT accuracy: {accuracy:.4f}")

### safety_eval

LLM-as-judge evaluation of the fine-tuned adapter's outputs.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "peft>=0.13",
        "accelerate",
        "openai",
        "mlflow",
        "nvtx",
    ],
)
def safety_eval(
    val: Input[Dataset],
    hf_adapter: Input[Model],
    base_model_id: str,
    judge_model_id: str,
    judge_system_prompt: str,
    judge_base_url: str,
    sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 256,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
):
    import json, pathlib, mlflow, nvtx, torch
    from openai import OpenAI

    val_data = json.loads(pathlib.Path(val.path).read_text())[:sample_size]

    # <<< UTILS_INJECT >>>

    # ---- USER CODE BLOCK ----
    # Load base model + HF PEFT adapter, run inference, score with judge
    # from transformers import AutoTokenizer, AutoModelForCausalLM
    # from peft import PeftModel
    # tokenizer = AutoTokenizer.from_pretrained(hf_adapter.path)
    # model = AutoModelForCausalLM.from_pretrained(
    #     base_model_id, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
    # model = PeftModel.from_pretrained(model, hf_adapter.path)
    # model.eval()
    # client = OpenAI(base_url=judge_base_url, api_key="ollama")
    scores = []
    avg_score = sum(scores) / len(scores) if scores else 0.0
    # ---- END USER CODE BLOCK ----

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-safety-eval"):
        mlflow.log_metric("safety_avg_score", avg_score)

    pathlib.Path(metrics.path).write_text(json.dumps({"safety_avg_score": avg_score}))
    print(f"Safety avg score: {avg_score:.4f}")

### deployment_gate

Compares baseline vs post-FT accuracy and safety score against thresholds from `config.yaml`.
On pass, writes `gate_result.json` to the shared PVC — same contract as `ft-eval` projects.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=[],
)
def deployment_gate(
    test: Input[Dataset],
    hf_adapter: Input[Model],
    baseline_metrics: Input[Metrics],
    postft_metrics: Input[Metrics],
    safety_metrics: Input[Metrics],
    baseline_safety_metrics: Input[Metrics],
    accuracy_delta_threshold: float,
    safety_score_threshold: float,
    safety_delta_threshold: float,
    run_id: str,
    pipeline_name: str = "",
    base_model_id: str = "",
    mlflow_experiment_name: str = "",
):
    import datetime, json, pathlib

    baseline        = json.loads(pathlib.Path(baseline_metrics.path).read_text())
    postft          = json.loads(pathlib.Path(postft_metrics.path).read_text())
    safety          = json.loads(pathlib.Path(safety_metrics.path).read_text())
    baseline_safety = json.loads(pathlib.Path(baseline_safety_metrics.path).read_text())

    baseline_acc          = baseline.get("baseline_accuracy", 0.0)
    postft_acc            = postft.get("postft_accuracy", 0.0)
    safety_score          = safety.get("safety_avg_score", 0.0)
    baseline_safety_score = baseline_safety.get("baseline_safety_avg_score", 0.0)

    acc_delta    = postft_acc - baseline_acc
    safety_delta = safety_score - baseline_safety_score
    passed = (
        acc_delta    >= accuracy_delta_threshold
        and safety_delta >= safety_delta_threshold
        and safety_score >= safety_score_threshold
    )

    print(f"Accuracy delta : {acc_delta:+.4f}  (threshold ≥ {accuracy_delta_threshold:.4f})")
    print(f"Safety delta   : {safety_delta:+.4f}  (threshold ≥ {safety_delta_threshold:.4f})")
    print(f"Safety score   : {safety_score:.4f}  (floor ≥ {safety_score_threshold:.4f})")
    print(f"Gate           : {'PASS' if passed else 'FAIL'}")

    if not passed:
        raise RuntimeError(
            f"Deployment gate failed — acc delta {acc_delta:+.4f}, "
            f"safety delta {safety_delta:+.4f}, safety score {safety_score:.4f}"
        )

    result = {
        "project": pipeline_name,
        "run_id": run_id,
        "base_model_id": base_model_id,
        "eval_passed": bool(passed),
        "safety_passed": bool(safety_score >= safety_score_threshold),
        "acc_delta": round(acc_delta, 6),
        "safety_score": round(safety_score, 6),
        "adapter_rel_path": f"nemo-adapters/{pipeline_name}/{run_id}",
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    }
    out_dir = pathlib.Path(f"/root/.cache/huggingface/runs/{pipeline_name}/{run_id}")
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "gate_result.json").write_text(json.dumps(result, indent=2))
    print(f"Gate result written: {out_dir}/gate_result.json")

### Pipeline

Wire the steps together. Defaults are read from `config.yaml` at import time.

In [ ]:
import yaml as _yaml, pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text())
_dataset_names = [d["name"] for d in _cfg["datasets"]]
_nemo_cfg = _cfg.get("nemo", {})
_profiling_cfg = _cfg.get("profiling", {})
_pipeline_name = "{{PROJECT_NAME}}"

from kfp import kubernetes as k8s_ext


@dsl.pipeline(name="{{PROJECT_NAME}}")
def pipeline(
    base_model_id: str = _cfg["model"]["hf_id"],
    nemo_model_id: str = _cfg["model"]["id"],
    nemo_base_url: str = _nemo_cfg.get("base_url", "http://nemo.test:8082"),
    nemo_namespace: str = _nemo_cfg.get("namespace", "default"),
    nemo_data_store_path: str = _nemo_cfg.get("data_store_path", "/data"),
    judge_model_id: str = _cfg["judge"]["model"],
    judge_system_prompt: str = _cfg["judge"]["system_prompt"],
    judge_base_url: str = _cfg["judge"]["base_url"],
    dataset_names: list = _dataset_names,
    num_epochs: int = _cfg["training"]["num_epochs"],
    learning_rate: float = _cfg["training"]["learning_rate"],
    batch_size: int = _cfg["training"]["batch_size"],
    val_size: float = _cfg["training"]["val_size"],
    test_size: float = _cfg["training"]["test_size"],
    eval_sample_size: int = _cfg["eval"]["sample_size"],
    safety_sample_size: int = _cfg["eval"]["safety_sample_size"],
    max_new_tokens: int = _cfg["eval"]["max_new_tokens"],
    safety_max_new_tokens: int = _cfg["eval"]["safety_max_new_tokens"],
    do_sample: bool = _cfg["eval"]["do_sample"],
    system_message: str = _cfg["eval"]["system_message"],
    accuracy_delta_threshold: float = _cfg["eval"]["accuracy_delta_threshold"],
    safety_score_threshold: float = _cfg["eval"]["safety_score_threshold"],
    safety_delta_threshold: float = _cfg["eval"]["safety_delta_threshold"],
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
    mlflow_experiment_name: str = "default",
):
    dl = download_model(base_model_id=base_model_id)

    prep = prepare_dataset(
        dataset_names=dataset_names,
        val_size=val_size,
        test_size=test_size,
    )
    prep.after(dl)

    base_eval = baseline_eval(
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        eval_sample_size=eval_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        max_new_tokens=max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
    )
    base_eval.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")
    base_eval.after(dl)

    base_safety = baseline_safety_eval(
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        judge_model_id=judge_model_id,
        judge_system_prompt=judge_system_prompt,
        judge_base_url=judge_base_url,
        sample_size=safety_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        max_new_tokens=safety_max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
    )
    base_safety.after(base_eval)
    base_safety.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")

    ft = fine_tune(
        train=prep.outputs["train_out"],
        base_model_id=nemo_model_id,
        nemo_base_url=nemo_base_url,
        nemo_namespace=nemo_namespace,
        nemo_data_store_path=nemo_data_store_path,
        num_epochs=num_epochs,
        learning_rate=learning_rate,
        batch_size=batch_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    # fine_tune does not need GPU — runs against NeMo Microservices API
    # but must wait for baseline evals to finish (GPU sequencing on single-node k3s)
    ft.after(base_safety)

    export = export_adapter(
        ft_model=ft.outputs["ft_model"],
        base_model_id=base_model_id,
    )
    export.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")

    post_eval = post_finetune_eval(
        val=prep.outputs["val_out"],
        hf_adapter=export.outputs["hf_adapter"],
        base_model_id=base_model_id,
        eval_sample_size=eval_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        max_new_tokens=max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
    )
    post_eval.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")

    safety = safety_eval(
        val=prep.outputs["val_out"],
        hf_adapter=export.outputs["hf_adapter"],
        base_model_id=base_model_id,
        judge_model_id=judge_model_id,
        judge_system_prompt=judge_system_prompt,
        judge_base_url=judge_base_url,
        sample_size=safety_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        max_new_tokens=safety_max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
    )
    safety.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")

    gate = deployment_gate(
        test=prep.outputs["test_out"],
        hf_adapter=export.outputs["hf_adapter"],
        baseline_metrics=base_eval.outputs["metrics"],
        postft_metrics=post_eval.outputs["metrics"],
        safety_metrics=safety.outputs["metrics"],
        baseline_safety_metrics=base_safety.outputs["metrics"],
        accuracy_delta_threshold=accuracy_delta_threshold,
        safety_score_threshold=safety_score_threshold,
        safety_delta_threshold=safety_delta_threshold,
        pipeline_name=_pipeline_name,
        base_model_id=base_model_id,
        run_id=run_id,
        mlflow_experiment_name=mlflow_experiment_name,
    )

    _SECRET = "mlabs-api-keys"
    _SECRET_KEYS = [
        "OPENAI_API_KEY", "HF_TOKEN", "ANTHROPIC_API_KEY",
        "WANDB_API_KEY", "LANGCHAIN_API_KEY", "NGC_API_KEY", "NVIDIA_API_KEY",
    ]
    _key_map = {k: k for k in _SECRET_KEYS}
    _HF_CACHE_PVC = "hf-model-cache"

    for _task in [dl, base_eval, base_safety, export, post_eval, safety, gate]:
        k8s_ext.mount_pvc(_task, pvc_name=_HF_CACHE_PVC, mount_path="/root/.cache/huggingface")

    for _task in [prep, dl, base_eval, base_safety, ft, export, post_eval, safety, gate]:
        k8s_ext.use_secret_as_env(_task, _SECRET, _key_map)

    # Nsight Operator GPU profiling — per-stage opt-in from config.yaml profiling: block.
    # A true value labels that stage pod nvidia-nsight-profile=enabled, so the operator
    # webhook injects nsys process-hook capture. /kfp-monitor then drives
    # ~/bin/nsight-export-report during the hot window to archive the report into
    # ~/shared/nsight/<project>/<run-id>/<stage>/. Never label the kubeflow namespace itself.
    # fine_tune is omitted — it has no local GPU pod (runs against the NeMo API).
    _profile_tasks = {
        "baseline-eval": base_eval,
        "baseline-safety-eval": base_safety,
        "export-adapter": export,
        "post-finetune-eval": post_eval,
        "safety-eval": safety,
    }
    for _stage, _task in _profile_tasks.items():
        if _profiling_cfg.get(_stage, False):
            k8s_ext.add_pod_label(_task, "nvidia-nsight-profile", "enabled")

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell.

What this does:
1. Reads `formatters.py`, `loaders.py`, `eval_helpers.py`, `utils.py` and inlines them into the component bodies
2. Copies all step cells and the pipeline cell
3. Writes `pipeline.py`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/build_pipeline.py"], check=True)

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (requires SSH tunnel).

In [ ]:
from kfp import compiler
from pipeline import pipeline

compiler.Compiler().compile(pipeline_func=pipeline, package_path="/tmp/pipeline.yaml")
print("Compiled: /tmp/pipeline.yaml")

In [ ]:
# Requires SSH tunnel: ssh -L 8080:localhost:8080 <user>@spark-79b7.local
import kfp

client = kfp.Client(host="http://localhost:8080")

In [ ]:
run = client.create_run_from_pipeline_package(
    pipeline_file="/tmp/pipeline.yaml",
    arguments={},
    run_name="notebook-run",
)
print(f"Run ID: {run.run_id}")

In [ ]:
import time

run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f"{state}")
    if state in ("SUCCEEDED", "FAILED", "CANCELED", "SKIPPED"):
        break
    time.sleep(30)